# RAG问答系统

## 课程概述
本课程旨在指导学员构建一个功能完善的多模态检索增强生成（RAG）问答系统。学员将学习如何整合本地知识库、实时网络搜索、以及特定网页内容抓取等多种信息源，通过大型语言模型实现智能问答。课程内容涵盖从文档加载、向量索引构建到多轮对话管理、外部工具集成和复杂指令处理等关键技术环节，最终打造一个能够理解用户意图并高效提供精准答案的AI应用。

* **课程时长**：1学时
* **适合人群**：AI开发者、内容创作者、产品经理
* **先修知识**：Python基础、机器学习基础、自然语言处理基本概念

**核心知识点**：

* **本地知识库构建与检索**：
    * 加载多种格式文档（PDF, DOCX, TXT）。
    * 文本切分与向量化（`RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`）。
    * 使用 `Qdrant` 构建和查询向量数据库。
* **检索增强生成 (RAG) 核心链**：
    * `MultiQueryRetriever` 提升检索相关性。
    * `ConversationalRetrievalChain` 实现带记忆的问答。
    * `ConversationBufferMemory` 管理多轮对话历史。
* **外部工具集成 (MCP)**：
    * 通过 `langchain_mcp_tools` 封装和调用MCP工具。
    * **Tavily**: 实现实时网络搜索能力。
    * **Fetch**: 实现指定URL内容的抓取和解析。
    * **Filesystem**: 实现本地文件读写操作。
* **Agent智能体构建 (LangGraph)**：
    * 使用 `create_react_agent` 构建能够自主规划和执行任务的AI智能体。
    * Agent对多工具的协同调用与任务拆解。

## 1. 项目背景

随着信息时代的飞速发展，人们对快速、准确获取知识和信息的需求日益增长。传统的检索引擎和单一知识库问答系统在处理复杂、多源、需要实时更新信息的问题时，往往显得力不从心。检索增强生成（RAG）技术通过结合预训练语言模型的强大生成能力与外部知识库的精准信息，为解决这一问题提供了有效途径。然而，现有的RAG系统仍面临诸多挑战。

### 1.1 当前RAG系统的痛点

尽管RAG技术取得了显著进展，但许多当前系统仍存在以下痛点：

1.  **信息来源单一与更新滞后**：多数RAG系统依赖于预先构建的本地知识库，难以接入和处理实时的外部信息（如最新新闻、研究论文、动态网页内容），导致回答可能基于过时或不全面的数据。
2.  **缺乏主动交互与执行能力**：传统RAG系统主要局限于“检索-生成”的被动响应模式，无法根据用户指令执行更复杂的操作，如定向抓取特定网页的最新内容、将分析结果保存到本地文件或与外部应用交互。
3.  **复杂任务处理能力不足**：面对需要多步骤推理、整合来自不同渠道信息（例如，本地文档、网页搜索结果、特定URL内容）才能解答的复杂问题，现有RAG系统往往难以有效协调和执行。
4.  **上下文理解与多轮对话的局限性**：在连续的多轮对话中，系统可能难以准确追踪用户意图的演变，尤其当对话涉及到需要调用不同工具或切换信息源时，上下文管理变得尤为复杂。
5.  **工具集成与扩展性受限**：将新的信息源或功能（如特定API、文件管理工具）集成到现有RAG流程中通常比较繁琐，缺乏标准化的协议和灵活的编排能力。

在这样的背景下，“**多源协作式智能RAG系统**”应运而生。它通过引入智能体（Agent）作为核心协调者，并借助模型上下文协议（MCP）标准化工具集成，旨在克服上述痛点。本系统能够动态编排包括本地知识库检索、实时网络搜索、定向内容抓取、文件系统操作在内的多种能力，从而更智能、更灵活地响应用户需求。

### 1.2 应用场景示例

该多模态RAG系统的核心优势在于其能够理解复杂指令，并智能调度多种工具来完成任务，以下是一些典型的应用场景：

1.  **智能研究助理**：
    * **用户指令**：“帮我查找关于‘强化学习在机器人路径规划中的最新应用’的论文，重点关注2024年以来的成果，总结主要方法和挑战，并将摘要保存到我的‘研究资料’目录下，文件名为‘强化学习路径规划进展.txt’。”
    * **系统执行流程**：
        1.  **理解指令**：Agent解析用户意图，识别出信息检索、内容分析、文件保存等多个子任务。
        2.  **网络搜索 (Tavily)**：调用Tavily搜索工具，查找关键词相关的最新论文链接和初步摘要。
        3.  **内容抓取 (Fetch)**：对于关键的论文链接（尤其是非PDF的网页介绍或博客），调用Fetch工具抓取详细内容。
        4.  **本地知识库问答 (RAG\_QA)**：结合本地可能已有的相关论文或文档进行内容补充。
        5.  **内容生成与总结**：Agent整合所有信息，交由LLM进行分析、总结主要方法与挑战。
        6.  **文件保存 (Filesystem)**：调用Filesystem工具，在用户指定的白名单路径下创建目录（如果不存在）并保存总结的TXT文件。

2.  **实时市场动态分析与报告**：
    * **用户指令**：“ICLR 2025的最佳论文是哪篇？请抓取其官方介绍页面的内容，提取核心贡献点，并结合Tavily搜索到的相关解读，生成一份简报。”
    * **系统执行流程**：
        1.  **初步搜索 (Tavily)**：Agent首先使用Tavily尝试找到ICLR 2025最佳论文的官方公告或相关新闻，获取论文标题和介绍页面URL。
        2.  **定向抓取 (Fetch)**：使用Fetch工具抓取该官方介绍页面的主要文本内容。
        3.  **内容分析 (LLM)**：LLM分析抓取到的内容，提取核心贡献。
        4.  **补充搜索 (Tavily)**：再次使用Tavily搜索关于这篇论文的解读、评论或分析文章。
        5.  **综合报告 (LLM + RAG\_QA)**：结合官方介绍和外部解读，生成一份综合简报。

3.  **多轮交互式信息探索与整理**：
    * **用户**：“我最近在学习Transformer模型，你能先给我一份它核心概念的简介吗？” (调用RAG\_QA)
    * **AI**：“...（生成简介）...”
    * **用户**：“很好。那它和传统的RNN相比，主要的优势和劣势有哪些？” (调用RAG\_QA，利用对话记忆)
    * **AI**：“...（生成对比分析）...”
    * **用户**：“请帮我找一篇关于‘Transformer中的注意力机制’的权威解释文章的链接。” (调用Tavily)
    * **AI**：“...（返回链接）...”
    * **用户**：“就这个链接，帮我阅读内容，并把它的关键点提取出来，保存成Markdown笔记。” (调用Fetch抓取内容，LLM提取关键点，Filesystem保存)
    * **AI**：“已将关键点提取并保存到 ‘Transformer注意力机制笔记.md’。”

通过智能体和MCP集成，RAG系统能够：

* **动态整合与调度**：根据用户需求，灵活组合和调用本地知识库、实时搜索引擎、网页抓取工具及文件管理工具。
* **复杂任务拆解与执行**：将用户的复杂指令分解为一系列可执行的子任务，并按逻辑顺序协调工具完成。
* **上下文感知与持续交互**：在多轮对话中保持对上下文的理解，确保后续交互的贯性和准确性。
* **主动信息获取与处理**：不仅仅是被动回答，更能主动获取、处理和管理信息，如自动下载、分析和保存用户指定的网络资源。

这不仅提升了 **信息获取的全面性、准确性和即时性** ，更极大改善了用户体验，使 **知识探索、研究分析和内容创作等任务** 变得更加 **智能化、自动化和高效化** 。

## 2. 项目分析与解决方案

本项目构建了一个多功能问答系统，其核心为基于LangGraph ReAct框架的**工具增强型单一智能体**，能整合本地知识、调用外部工具以理解和执行复杂任务。

### 2.1 系统架构设计

系统采用**工具增强的单一智能体架构**，主要组件包括：

1.  **ReAct智能体 (LangGraph)**：基于`create_react_agent`构建，作为核心控制器，负责任务规划、工具选择与调用、以及最终响应的生成。
2.  **大型语言模型 (LLM)**：为智能体提供核心的推理、理解和内容生成能力。
3.  **本地RAG问答工具 (`rag_tool`)**: 自定义工具，封装了完整的本地知识检索与生成流程，包括：多种格式文档（PDF, DOCX, TXT）加载、文本分割、基于`Qdrant`和`OpenAIEmbeddings`的向量检索，以及利用`ConversationalRetrievalChain`（含`MultiQueryRetriever`和`ConversationBufferMemory`）实现的上下文感知问答。
4.  **外部工具集 (通过MCP集成)**：通过`langchain_mcp_tools`标准化接入外部服务，包括：
    * **Tavily**: 用于实时在线搜索。
    * **Fetch**: 用于抓取指定URL的网页或PDF内容。
    * **Filesystem**: 用于本地文件系统的读写操作。
5.  **记忆与状态管理**:
    * **Agent状态检查点 (`InMemorySaver`)**: 记录和管理智能体在多轮对话中的会话状态（基于`thread_id`）。
    * **RAG对话记忆 (`ConversationBufferMemory`)**: 在`rag_tool`内部维护本地知识库问答的特定上下文。

架构如下图所示 (Mermaid):


In [ ]:
import os
from IPython.display import SVG, display

mermaid_code = """
graph TD
    User[用户输入] --> Agent[ReAct智能体 LangGraph+LLM]
    Agent -- 进行工具规划与选择 --> ToolsSubgraph[工具集]
    subgraph ToolsSubgraph
        direction TB
        LocalRAG[本地RAG工具 rag_tool]
        TavilySearch[Tavily实时搜索 MCP]
        FetchURL[Fetch URL内容 MCP]
        FileSystemOps[文件系统操作 MCP]
    end
    ToolsSubgraph -- 工具执行结果/观察 --> Agent
    Agent -- 生成最终响应 --> Output[系统输出]
    Agent -- 管理会话状态 --> AgentState[InMemorySaver 检查点]
    LocalRAG -- 访问/检索 --> VectorDB[Qdrant向量数据库]
    LocalRAG -- 管理RAG对话上下文 --> RAGMemory[ConversationBufferMemory]
"""

with open("diagram.mmd", "w", encoding="utf-8") as f:
    f.write(mermaid_code)

os.system("mmdc -i diagram.mmd -o diagram.svg")

display(SVG(filename="diagram.svg"))

### 2.2 任务处理流程

系统通过ReAct (Reason+Act) 循环处理用户请求：

1.  **输入与理解**: 智能体接收用户输入，LLM分析用户意图，并结合上下文信息规划执行步骤。
2.  **工具选择与执行**: 若判断需要工具辅助，智能体根据预设的工具描述和当前任务，选择最合适的工具（如`rag_tool`进行本地知识查询，Tavily进行网络搜索等）并执行。工具执行后将结果（观察）返回给智能体。
3.  **迭代与响应**: 智能体根据工具返回的观察结果，进行新一轮的“思考-行动”迭代，直至任务目标达成。最终，智能体整合信息并生成答复给用户。
4.  **状态维护**: `InMemorySaver`负责维护整个Agent的会话历史和状态，而`rag_tool`内的`ConversationBufferMemory`则保障RAG查询的上下文连贯性，共同支持流畅的多轮对话。

### 2.3 关键技术点

1.  **ReAct智能体 (LangGraph)**: 核心是`create_react_agent`构建的能够自主规划、调用工具的智能体，其效能依赖于LLM的强大推理能力和对工具功能的准确描述。
2.  **本地RAG流程 (`rag_tool`)**: 高效整合了从多格式文档处理、`Qdrant`语义检索、多查询扩展到上下文感知问答的完整流程，充分利用本地知识库。
3.  **MCP模块化工具集成**: 利用`langchain_mcp_tools`实现了对Tavily（搜索）、Fetch（网页抓取）、Filesystem（文件操作）等外部能力的标准化、可插拔式集成，增强了系统的功能和扩展性。
4.  **多轮对话与状态管理**: 通过`InMemorySaver`（Agent全局会话）和`ConversationBufferMemory`（RAG局部上下文）的组合，有效管理对话状态，确保交互的连贯性和准确性。
5.  **异步任务执行**: 广泛使用`asyncio`进行异步编程，优化了I/O密集型操作（如网络请求、文件访问）的执行效率，提升了系统的整体响应速度。


## 3. 项目实现

下面将通过Jupyter Notebook的形式展示项目的实现过程。



In [ ]:
# 导入必要的库
import os, asyncio  # os：与操作系统交互；asyncio：用于异步任务处理
from dotenv import load_dotenv  # load_dotenv：加载.env文件中的环境变量
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader  # 导入不同文件格式的文档加载器（PDF、DOCX、纯文本）
from langchain.text_splitter import RecursiveCharacterTextSplitter  # 导入文本分割器，用于将文本按字符数分割成块
from langchain_community.vectorstores import Qdrant  # Qdrant：一个向量存储，用于存储和搜索向量
from qdrant_client import QdrantClient  # QdrantClient：与Qdrant向量存储交互的客户端库
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAIEmbeddings：用于生成向量嵌入；ChatOpenAI：用于集成OpenAI的聊天模型
from langchain.retrievers.multi_query import MultiQueryRetriever  # MultiQueryRetriever：同时检索多个查询的工具
from langchain.memory import ConversationBufferMemory  # ConversationBufferMemory：用于存储对话状态的内存
from langchain.chains import ConversationalRetrievalChain  # ConversationalRetrievalChain：构建一个处理对话检索的链
from langchain.tools import Tool  # Tool：允许向链中添加自定义工具
from langchain_mcp_tools import convert_mcp_to_langchain_tools  # 将MCP工具转换为LangChain兼容的工具
from langgraph.prebuilt import create_react_agent  # create_react_agent：创建一个反应式的决策代理
from langgraph.checkpoint.memory import InMemorySaver  # InMemorySaver：用于将代理状态保存到内存中

# 加载.env文件中的环境变量（用于安全管理API密钥和配置）
load_dotenv()

# 设置OPENAI API的基础URL和API密钥
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_API_BASE")  # 从环境中获取OPENAI_API_BASE
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")  # 从环境中获取OPENAI_API_KEY

In [ ]:
# 数据提取部分
import os  # os：与操作系统交互，方便进行文件操作
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader  # 导入文档加载器，用于加载不同格式的文件（PDF、DOCX、TXT）

# 设置文件夹路径，存放需要加载的文档
base_dir = 'Docs'  # 设置文档所在的文件夹路径
documents = []  # 创建一个空列表，用于存储加载后的文档内容

# 遍历文件夹中的每个文件，根据文件类型选择合适的加载器
for file in os.listdir(base_dir):  # os.listdir(base_dir)：列出指定文件夹中的所有文件
    file_path = os.path.join(base_dir, file)  # 获取文件的完整路径
    if file.endswith('.pdf'):  # 如果文件是PDF格式
        loader = PyPDFLoader(file_path)  # 使用PyPDFLoader加载PDF文件
        documents.extend(loader.load())  # 将加载的文档内容添加到documents列表中
    elif file.endswith('.docx'):  # 如果文件是DOCX格式
        loader = Docx2txtLoader(file_path)  # 使用Docx2txtLoader加载DOCX文件
        documents.extend(loader.load())  # 将加载的文档内容添加到documents列表中
    elif file.endswith('.txt'):  # 如果文件是TXT格式
        loader = TextLoader(file_path)  # 使用TextLoader加载TXT文件
        documents.extend(loader.load())  # 将加载的文档内容添加到documents列表中


# 文本分割部分
from langchain.text_splitter import RecursiveCharacterTextSplitter  # 导入递归字符分割器，用于将文档分割成多个小块

# 创建一个文本分割器实例，设置每个文本块的大小为200个字符，重叠部分为10个字符
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)

# 将加载的文档进行分割，每个分割出的文档块长度为200，且块之间有10个字符的重叠
chunked_documents = text_splitter.split_documents(documents)  # 将分割后的文本块存储到chunked_documents列表中

In [ ]:
# 嵌入、向量入库部分
from langchain_community.vectorstores import Qdrant  # 导入Qdrant向量存储模块，用于将文档转换为向量并存储
from qdrant_client import QdrantClient  # 导入Qdrant客户端库，用于与Qdrant服务进行交互
from langchain_openai import OpenAIEmbeddings  # 导入OpenAI嵌入模型，用于生成文档的向量表示

# 检查是否存在Docs-database文件夹，如果不存在则创建新的Qdrant向量存储
if not os.path.exists("./Docs-database"):  # 判断文件夹是否存在
    # 使用Qdrant将分割后的文档转换为向量并存入向量存储
    vectorstore = Qdrant.from_documents(
        documents=chunked_documents,  # 使用分割后的文档
        embedding=OpenAIEmbeddings(),  # 使用OpenAI的嵌入模型生成文档的向量表示
        path="./Docs-database",  # 设置向量存储的路径
        collection_name="my_documents"  # 设置存储的集合名称
    )
else:
    # 如果向量存储已经存在，则通过QdrantClient连接到现有的存储
    client = QdrantClient(path="./Docs-database")  # 创建Qdrant客户端，连接到现有的存储
    vectorstore = Qdrant(
        client=client,  # 使用客户端连接到Qdrant
        collection_name="my_documents",  # 设置集合名称
        embeddings=OpenAIEmbeddings()  # 使用OpenAI的嵌入模型生成文档的向量表示
    )


In [ ]:
# 准备模型和Retrieval链
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from langgraph.checkpoint.memory import InMemorySaver
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.tools import Tool

# 初始化大语言模型
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 构建多查询检索器：通过 LLM 自动扩展用户问题，提升召回率
retriever = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(), llm=llm)

# 创建对话记忆模块：用于存储历史对话，实现多轮上下文理解
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
memory.clear()

# 构建 RAG 问答链：结合检索器和记忆，实现上下文增强的智能问答
qa_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, verbose=True)

# 封装为工具：便于 Agent 调用该问答功能
rag_tool = Tool(
    name="RAG_QA",
    func=lambda q: qa_chain({"question": q})["answer"],
    description="对一般知识类问题，先检索文档再回答"
)

In [ ]:
# 导入创建反应式代理的工具
from langgraph.prebuilt import create_react_agent  # 导入LangGraph的工具，用于创建反应式代理

# 定义一个异步函数，用于处理用户的输入消息
async def ask(msg):
    tools = []  # 初始化一个空的工具列表，用于保存代理可用的工具
    tools.append(rag_tool)  # 将rag_tool（假设这是一个已定义的工具）添加到工具列表中
    
    # 创建一个反应式代理(agent)，该代理能够调用指定的工具（这里是rag_tool）
    # llm：假设这是一个已经初始化的语言模型
    # checkpointer=InMemorySaver()：定义内存检查点用于保存代理的状态
    agent = create_react_agent(llm, tools, checkpointer=InMemorySaver())  # 创建代理
    
    # 使用代理的ainvoke方法异步处理用户的输入消息（msg）
    # config={"thread_id": "session-001"}：指定会话的ID，用于区分不同的会话
    res = await agent.ainvoke({"messages": msg}, config={"thread_id": "session-001"})
    
    # 返回代理处理结果中的消息部分
    return res["messages"]

# 调用ask函数，询问关于PoLMs的问题
await ask("什么是PoLMs？")

In [ ]:
# # tavily_mcp.py - 将 Tavily 搜索能力封装为独立 MCP 工具

# from mcp.server.fastmcp import FastMCP         # 引入 FastMCP，用于构建标准化的 MCP 工具服务器
# from tavily import TavilyClient                # 引入 Tavily SDK，用于访问在线搜索 API
# from dotenv import load_dotenv                 # 加载环境变量
# import os
# load_dotenv()                                  # 从 .env 文件中加载 Tavily API 密钥
# tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))  # 初始化 Tavily 客户端
# mcp = FastMCP("Tavily")                        # 注册一个名为 "Tavily" 的 MCP 工具服务

# @mcp.tool() # 定义 MCP 工具函数：search
# async def search(query: str, search_depth: str = "basic") -> str:
#     """
#     输入：query（搜索关键词），search_depth（搜索深度）
#     输出：格式化后的搜索结果字符串
#     功能：调用 Tavily API 执行网页搜索，并提取标题、内容与链接返回
#     """
#     try:
#         results = tavily.search(query=query, search_depth=search_depth, max_results=5)
#         formatted = []
#         for r in results.get("results", []):
#             formatted.append(
#                 f"标题：{r.get('title', '无标题')}\n"
#                 f"内容：{r.get('content', '无内容')}\n"
#                 f"链接：{r.get('url', '无链接')}\n"
#             )
#         return "\n---\n".join(formatted) if formatted else "未找到相关结果"
#     except Exception as e:
#         return f"搜索出错：{str(e)}"

# # 启动 MCP 工具服务（使用 stdio 协议）
# if __name__ == "__main__":
#     mcp.run(transport="stdio")

In [ ]:
# 定义一个字典mcp_configs，用于存储不同MCP工具的配置信息
mcp_configs = {
    # 配置Tavily的MCP
    "tavily": {
        "command": "python",  # 使用python命令运行Tavily相关的脚本
        "args": ["tavily_mcp.py"],  # 传递的参数，指定执行的脚本是tavily_mcp.py
        "transport": "stdio"  # 指定与Tavily MCP通信的传输方式为stdio（标准输入输出）
    },
    
    # 配置Fetch的MCP
    "fetch": {
        "command": "uvx",  # 使用uvx命令来运行Fetch MCP
        "args": ["mcp-server-fetch"]  # 传递的参数，指定Fetch服务器的启动命令
    },
    
    # 配置文件系统的MCP
    "filesystem": {
        "command": "npx",  # 使用npx命令执行文件系统的MCP
        "args": [
            "-y",  # 自动接受安装依赖
            "@modelcontextprotocol/server-filesystem",  # 使用modelcontextprotocol的文件系统服务
            "/Users/orzjh/Documents/LangChain",  # 设置文件系统的根路径
        ]
    },
}

In [9]:
# 导入所需的模块
from langchain_mcp_tools import convert_mcp_to_langchain_tools  # 将MCP工具转换为LangChain兼容的工具
from langgraph.prebuilt import create_react_agent  # 创建一个反应式代理，用于处理用户请求
from langgraph.checkpoint.memory import InMemorySaver  # 用于保存代理的状态

# 异步主函数
async def main():
    # convert_mcp_to_langchain_tools 会启动每个MCP服务器，并将它们封装成异步工具对象
    llm = ChatOpenAI(model_name="deepseek-v3", temperature=0)  # 初始化聊天模型，使用OpenAI的deepseek-v3模型，温度设置为0（确定性回答）
    
    # 将MCP工具配置转换为LangChain兼容的工具，并启动相应的MCP服务器
    tools, cleanup = await convert_mcp_to_langchain_tools(mcp_configs)
    
    # 将额外的工具添加到工具列表中，这里假设rag_tool是一个预定义的工具
    tools.append(rag_tool)
    
    # 创建一个反应式代理，该代理将使用指定的LLM和工具来处理用户的请求
    agent = create_react_agent(llm, tools, checkpointer=InMemorySaver())  # 使用InMemorySaver来保存代理的状态
    
    # 无限循环等待用户输入
    while True:
        # 获取用户输入的消息
        msg = input("You: ")
        
        # 如果用户输入 "exit" 或 "quit" 则退出对话
        if msg.lower() in ("exit", "quit"):
            print("再见！")  # 向用户显示再见信息
            break
        
        # 输出用户输入的消息
        print("User:", msg)
        
        # 使用代理异步调用，传递消息并获取AI的响应
        res = await agent.ainvoke({"messages": msg}, config={"thread_id": "session-001"})
        
        # 输出AI的响应消息
        print("AI response:", res["messages"][-1].content)

    # 清理MCP工具和资源
    await cleanup()

await main()

In [ ]:
# # 导入所需的模块
# from langchain_mcp_tools import convert_mcp_to_langchain_tools  # 将MCP工具转换为LangChain兼容的工具
# from langgraph.prebuilt import create_react_agent  # 创建一个反应式代理，用于处理用户请求
# from langgraph.checkpoint.memory import InMemorySaver  # 用于保存代理的状态

# # 异步主函数
# async def main():
#     # convert_mcp_to_langchain_tools 会启动每个MCP服务器，并将它们封装成异步工具对象
#     llm = ChatOpenAI(model_name="deepseek-v3", temperature=0)  # 初始化聊天模型，使用OpenAI的deepseek-v3模型，温度设置为0（确定性回答）
    
#     # 将MCP工具配置转换为LangChain兼容的工具，并启动相应的MCP服务器
#     tools, cleanup = await convert_mcp_to_langchain_tools(mcp_configs)
    
#     # 将额外的工具添加到工具列表中，这里假设rag_tool是一个预定义的工具
#     tools.append(rag_tool)
    
#     # 创建一个反应式代理，该代理将使用指定的LLM和工具来处理用户的请求
#     agent = create_react_agent(llm, tools, checkpointer=InMemorySaver())  # 使用InMemorySaver来保存代理的状态
    
#     # 无限循环等待用户输入
#     while True:
#         # 获取用户输入的消息
#         msg = input("You: ")
        
#         # 如果用户输入 "exit" 或 "quit" 则退出对话
#         if msg.lower() in ("exit", "quit"):
#             print("再见！")  # 向用户显示再见信息
#             break
        
#         # 输出用户输入的消息
#         print("User:", msg)
        
#         # 使用代理异步调用，传递消息并获取AI的响应
#         res = await agent.ainvoke({"messages": msg}, config={"thread_id": "session-001"})
        
#         # 输出AI的响应消息
#         print("AI response:", res["messages"][-1].content)

#     # 清理MCP工具和资源
#     await cleanup()

# await main()

In [ ]:
# ICLR 2025的best paper标题是什么？
# 抓取https://blog.iclr.cc/2025/04/22/announcing-the-outstanding-paper-awards-at-iclr-2025/的内容，转成txt格式并保存到Docs文件夹下。
# 我刚才都说了什么？